# Predictions, plots, and derivatives

Follow [setup](README.md), then run cells in order. This tutorial evaluates a bundled checkpoint for one amplitude; it uses the saved model directly.

In [ ]:
%matplotlib inline
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from modpinn.training import load_pretrained, evaluate_pretrained

torch.set_num_threads(1)
experiment = "best/amplitude-0.1"
model = load_pretrained(experiment)
metrics = evaluate_pretrained(experiment)
assert all(torch.isfinite(torch.tensor(value)) for value in metrics.values())
print(metrics)

## Predict a radial slice

Input columns: time, radius. Outputs: scalar field, lapse, compactness, each shaped `(N, 1)`. The example samples time 2 within the reference domain.

In [ ]:
radius = torch.linspace(0.01, 10.01, 256, dtype=model.DTYPE)
points = torch.stack((torch.full_like(radius, 2.0), radius), dim=1)
with torch.no_grad():
    fields = model(points)
assert all(field.shape == (256, 1) and torch.isfinite(field).all() for field in fields)

labels = ["Scalar field", "Lapse", "Compactness"]
with plt.rc_context({"font.family": "STIXGeneral", "font.size": 11, "axes.axisbelow": True}):
    fig, axes = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")
    for ax, field, label in zip(axes, fields, labels):
        ax.plot(radius.numpy(), field[:, 0].numpy(), color="#0072B2", linewidth=1.8)
        ax.set(xlabel="Radius r", title=label, xlim=(0.01, 10.01))
    display(fig)
    plt.close(fig)

## Differentiate

Enable coordinate gradients and omit `torch.no_grad()`. Derivative columns correspond to time and radius.

In [ ]:
query = torch.tensor([[0.1, 0.2], [1.0, 0.8]], dtype=model.DTYPE, requires_grad=True)
phi, alpha, compactness = model(query)
gradient = torch.autograd.grad(phi.sum(), query, create_graph=True)[0]
second_time = torch.autograd.grad(gradient[:, 0].sum(), query)[0][:, 0]
assert gradient.shape == (2, 2)
assert torch.isfinite(gradient).all() and torch.isfinite(second_time).all()
print("Columns: dphi/dt, dphi/dr")
print(gradient.detach())
print("Second time derivatives:", second_time.detach())

## Interpretation

Plots show raw predictions. See [results](../results.md) for error norms and clipping; use reference comparisons to assess accuracy. Continue with [training](quickstart.ipynb) or [saved-model inference](../inference.md).